# Open Data Exploration: X-Maze Behavior and Spatial Firing

In this notebook, we will use an open NWB dataset from DANDI to explore behavior, spikes, and spatial firing. The dataset comes from Aery Jones et al. (2026), where mice ran an X-maze spatial match/non-match to position task while Neuropixels probes recorded from medial entorhinal cortex (MEC) and CA1.

## 0. Setup

Run this once. In Colab, missing packages will be installed into the active runtime.

In [ ]:
import importlib
import subprocess
import sys

PACKAGE_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "pillow": "PIL",
    "seaborn": "seaborn",
    "dandi": "dandi",
    "pynwb": "pynwb",
    "remfile": "remfile",
    "h5py": "h5py",
    "h5glance": "h5glance",
    "plotly": "plotly",
}

import importlib
import tutorial_utils

importlib.reload(tutorial_utils)

from tutorial_utils import (
    EXPECTED_NWB_PATHS,
    check_nwb_path,
    get_unit_spike_times as fetch_unit_spike_times,
    unit_region_from_row,
)


failed = {}
for package, import_name in PACKAGE_IMPORTS.items():
    try:
        importlib.import_module(import_name)
    except Exception as exc:
        failed[package] = repr(exc)

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

if failed:
    print("Installing missing packages into this notebook kernel:", list(failed))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *failed])
    print("Installed. If imports still fail, restart the runtime and rerun this cell.")
else:
    print("All required packages are available in this notebook kernel.")

## 1. Plotting Defaults

Use inline high-resolution plots and a modern color-blind-aware palette. The plotting context is set to `notebook`, which keeps figures readable without oversized text.

In [ ]:
%config InlineBackend.figure_format = "retina" #high-resolution

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

COLORS = {
    "blue": "#0072B2",
    "sky": "#56B4E9",
    "green": "#009E73",
    "orange": "#E69F00",
    "vermillion": "#D55E00",
    "purple": "#7A68A6",
    "pink": "#CC79A7",
    "gray": "#5F6368",
    "light_gray": "#D8DEE9",
    "dark": "#23272F",
}
PALETTE = [COLORS[k] for k in ["blue", "orange", "green", "pink", "sky", "vermillion", "purple", "gray"]]

sns.set_theme(
    context="notebook",
    style="ticks",
    palette=PALETTE,
    rc={
        "figure.figsize": (7, 4.5),
        "figure.dpi": 180,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "axes.titleweight": "bold",
        "axes.labelcolor": COLORS["dark"],
        "text.color": COLORS["dark"],
        "savefig.bbox": "tight",
    },
)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=PALETTE)

## 2. Stream One NWB File From DANDI

We will start with one file from DANDI dataset `001701`. This dataset contains Neuropixels recordings from medial entorhinal cortex (MEC) and hippocampal area CA1. 

Paper context:

- Mice ran an X-maze with rewarded ports at arm ends.
- Days 1-10 used a match-to-sample rule.
- Days 11-20 used a nonmatch-to-sample rule.
- The paper asks how MEC represents task-relevant remote locations, especially during immobility.

In [ ]:
from dandi.dandiapi import DandiAPIClient
import h5py
import remfile
from pynwb import NWBHDF5IO

DANDISET_ID = "001701"
DANDISET_VERSION = "0.260120.0303"

# Change this later to explore another animal or day.
asset_path = "sub-Lamarr/sub-Lamarr_ses-Lamarr-DY01-g0_behavior+ecephys.nwb"

client = DandiAPIClient()
dandiset = client.get_dandiset(DANDISET_ID, DANDISET_VERSION)
asset = dandiset.get_asset_by_path(asset_path)

stream_url = asset.get_content_url(follow_redirects=1)
remote_file = remfile.File(stream_url)
h5_file = h5py.File(remote_file, mode="r")
io = NWBHDF5IO(file=h5_file, mode="r", load_namespaces=False)
nwb = io.read()

print("Streaming NWB file:")
print(asset_path)
print("\nSession description:", nwb.session_description)
print("Identifier:", nwb.identifier)

## 3. Explore The NWB File Structure

NWB files are structured hierarchically. Before analyzing, find where the important data live.

**Activity: Use the browser below to find paths for position, units, and LFP.**

In [ ]:
from h5glance import H5Glance

H5Glance(h5_file)

### Fill In The Paths

Use the browser above to fill these in before running the check cell.

| Data | Path |
| --- | --- |
| Position | `_____` |
| Units | `_____` |
| LFP | `_____` |

In [ ]:
# Fill in the paths you found. These are strings, so keep the quotation marks.
paths = {
    "position": "processing/behavior/Position/position",
    "units": "units",
    "lfp": "processing/probe_0_channel_55/LFP",
}

for name, path in paths.items():
    status = "Correct!" if check_nwb_path(h5_file, path, **EXPECTED_NWB_PATHS[name]) else "Try again :("
    print(f"{name:15s} {status:12s} {path}")

The file browser shows where different objects live inside the NWB file. PyNWB is a Python library that reads NWB files and represents those objects as Python objects rather than raw HDF5 groups and datasets.

In the next cell, we use the paths you found to retrieve the position object and the units table from the streamed file. The small helper converts a path such as `processing/behavior/Position/position` into step-by-step access through the `nwb` object.


In [ ]:
def get_by_hdf5_path(nwb_object, path):
    """Convert a simple HDF5-style path into chained NWB object access."""
    obj = nwb_object
    for part in path.split("/"):
        if not part:
            continue
        obj = obj[part] if hasattr(obj, "__getitem__") else getattr(obj, part)
    return obj

position_series = get_by_hdf5_path(nwb, paths["position"])
units = get_by_hdf5_path(nwb, paths["units"])

print("Position object:", position_series)
print("--------------")
print("\nNumber of units:", len(units))

## 4. Behavior


The position object stores the animal's tracked location over time.

In the next cell, we will:

- read the full position array from the NWB file,
- check how many samples and position dimensions it contains,
- reconstruct a time value for each sample using the sampling rate and start time,
- split the two position columns into `x` and `y`,
- put `time`, `x`, and `y` into a `DataFrame` so the data are easy to inspect and plot.

The idea is to examine position and time together: we want to know **when** an animal was at each x and y coordinate. 


In [ ]:
position_data = position_series.data[:]  # fetch data from the position directory of the NWB file
sampling_rate = position_series.rate
series_start_time = position_series.starting_time

# Question: how do we reconstruct the time point for each position sample?
# Hint: sample 0 occurs at series_start_time, and samples are spaced 1 / sampling_rate seconds apart.
# Solved version:
raw_time = series_start_time + np.arange(position_data.shape[0]) / sampling_rate
raw_x = position_data[:, 0]
raw_y = position_data[:, 1]

# Some sessions contain missing tracking samples marked as NaN.
# Remove them once here, so downstream histograms and interpolations use finite positions.
valid_position = np.isfinite(raw_time) & np.isfinite(raw_x) & np.isfinite(raw_y)
time = raw_time[valid_position]
x = raw_x[valid_position]
y = raw_y[valid_position]

position_df = pd.DataFrame({"time": time, "x": x, "y": y})

print("position_data.shape:", position_data.shape)
print("sampling_rate:", sampling_rate, "Hz")
print("valid position samples:", f"{valid_position.sum()} / {len(valid_position)}")
print("time range:", f"{time[0]:.2f} to {time[-1]:.2f} s")
display(position_df.head())

Now we plot the full x-y position trace.

After this overview, choose a short time window from the printed time range above and plot that segment on top of the full trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(position_df["x"], position_df["y"], color=COLORS["light_gray"], linewidth=0.8)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x position")
ax.set_ylabel("y position")
plt.show()

In [ ]:
# Question: choose a short time window to inspect.
# Use the printed time range above, then edit these two values.
window_start = 90
window_stop = 120

window_df = position_df.query("@window_start <= time <= @window_stop")
if window_df.empty:
    raise ValueError("No position samples found in this window. Choose start and stop times within the printed time range.")

print(f"Selected {len(window_df)} samples from {window_start} to {window_stop} seconds.")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(position_df["x"], position_df["y"], color=COLORS["light_gray"], linewidth=0.8, alpha=0.8, label="full session")
ax.plot(window_df["x"], window_df["y"], color=COLORS["blue"], linewidth=2.0, label=f"{window_start}-{window_stop} s")
ax.scatter(window_df["x"].iloc[0], window_df["y"].iloc[0], color=COLORS["green"], s=45, zorder=3, label="start")
ax.scatter(window_df["x"].iloc[-1], window_df["y"].iloc[-1], color=COLORS["orange"], s=45, zorder=3, label="stop")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x position (cm)")
ax.set_ylabel("y position (cm)")
ax.legend(frameon=False, loc="best")
plt.show()

## 5. Movement And Pausing

Position gives us more than the shape of the maze. From x-y position over time, we can derive behavioral variables such as speed and acceleration.

These variables help us ask richer questions later, for example:

- does neural firing differ when the animal is moving versus stationary?
- does theta-band LFP power increase with running speed?
- are rapid changes in movement visible in acceleration?

Speed is distance traveled per unit time:

```text
speed = distance / time
```

For x-y position, the distance between consecutive samples is:

```text
distance = sqrt(dx^2 + dy^2)
```

In [ ]:
# Fill-in challenge: what should go into dx, dy, and dt?
dx = np.diff(x)
dy = np.diff(y)
dt = np.diff(time)

speed = np.sqrt(dx**2 + dy**2) / dt
speed_time = time[:-1] + dt / 2

speed_df = pd.DataFrame({"time": speed_time, "speed": speed})
display(speed_df.head())

Now plot speed over a time period

In [ ]:
# Question: choose a time window for the speed trace.
# Use the same time range as the position data, then edit these values.
speed_window_start = 90
speed_window_stop = 120

speed_window_df = speed_df.query("@speed_window_start <= time <= @speed_window_stop")
if speed_window_df.empty:
    raise ValueError("No speed samples found in this window. Choose start and stop times within the session time range.")

print(f"Selected {len(speed_window_df)} speed samples from {speed_window_start} to {speed_window_stop} seconds.")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(speed_window_df["time"], speed_window_df["speed"], color=COLORS["blue"], linewidth=0.9)
ax.set_xlabel("time (s)")
ax.set_ylabel("speed (cm/s)")
sns.despine(ax=ax)
plt.show()

We can also look at speed and position together:

In [ ]:
# Combine position and speed for the selected window.
# Speed was computed between consecutive position samples, so we use midpoint positions.
speed_position_df = speed_df.copy()
speed_position_df["x"] = (x[:-1] + x[1:]) / 2
speed_position_df["y"] = (y[:-1] + y[1:]) / 2

speed_position_window_df = speed_position_df.query(
    "@speed_window_start <= time <= @speed_window_stop"
)
if speed_position_window_df.empty:
    raise ValueError("No samples found in this window. Choose start and stop times within the session time range.")

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(position_df["x"], position_df["y"], color=COLORS["light_gray"], linewidth=0.8, alpha=0.8)
sc = ax.scatter(
    speed_position_window_df["x"],
    speed_position_window_df["y"],
    c=speed_position_window_df["speed"],
    s=14,
    cmap="viridis",
    zorder=3,
)
ax.scatter(
    speed_position_window_df["x"].iloc[0],
    speed_position_window_df["y"].iloc[0],
    color=COLORS["green"],
    edgecolor="white",
    linewidth=0.8,
    s=70,
    zorder=4,
    label="start",
)
ax.scatter(
    speed_position_window_df["x"].iloc[-1],
    speed_position_window_df["y"].iloc[-1],
    color=COLORS["orange"],
    edgecolor="white",
    linewidth=0.8,
    s=70,
    zorder=4,
    label="stop",
)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x position")
ax.set_ylabel("y position")
plt.colorbar(sc, ax=ax, label="speed (cm/s)")
ax.legend(frameon=False, loc="best")
sns.despine(ax=ax)
plt.show()

We can also derive acceleration from speed.

Acceleration is the change in speed per unit time:

```text
acceleration = change in speed / change in time
```

Large positive values mean the animal is speeding up. Large negative values mean the animal is slowing down.

In [ ]:
# Fill-in challenge: how should acceleration be computed from speed?
d_speed = np.diff(speed)
d_speed_time = np.diff(speed_time)

acceleration = d_speed / d_speed_time
acceleration_time = speed_time[:-1] + d_speed_time / 2

acceleration_df = pd.DataFrame({"time": acceleration_time, "acceleration": acceleration})
display(acceleration_df.head())

Now plot acceleration over a selected time window.

In [ ]:
# Use the same window as the speed plot by default, or edit these values.
acceleration_window_start = speed_window_start
acceleration_window_stop = speed_window_stop

acceleration_window_df = acceleration_df.query(
    "@acceleration_window_start <= time <= @acceleration_window_stop"
)
if acceleration_window_df.empty:
    raise ValueError("No acceleration samples found in this window. Choose start and stop times within the session time range.")

print(f"Selected {len(acceleration_window_df)} acceleration samples from {acceleration_window_start} to {acceleration_window_stop} seconds.")

fig, ax = plt.subplots(figsize=(9, 3))
ax.axhline(0, color=COLORS["gray"], linewidth=0.8, alpha=0.6)
ax.plot(acceleration_window_df["time"], acceleration_window_df["acceleration"], color=COLORS["vermillion"], linewidth=0.9)
ax.set_xlabel("time (s)")
ax.set_ylabel("acceleration (cm/s^2)")
sns.despine(ax=ax)
plt.show()

We can also look at acceleration and position together.

In [ ]:
# Acceleration was computed between consecutive speed samples, so use midpoint positions again.
acceleration_position_df = acceleration_df.copy()
speed_x = (x[:-1] + x[1:]) / 2
speed_y = (y[:-1] + y[1:]) / 2
acceleration_position_df["x"] = (speed_x[:-1] + speed_x[1:]) / 2
acceleration_position_df["y"] = (speed_y[:-1] + speed_y[1:]) / 2

acceleration_position_window_df = acceleration_position_df.query(
    "@acceleration_window_start <= time <= @acceleration_window_stop"
)
if acceleration_position_window_df.empty:
    raise ValueError("No samples found in this window. Choose start and stop times within the session time range.")

accel_abs_max = np.nanpercentile(np.abs(acceleration_position_window_df["acceleration"]), 95)
if accel_abs_max == 0:
    accel_abs_max = 1

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(position_df["x"], position_df["y"], color=COLORS["light_gray"], linewidth=0.8, alpha=0.8)
sc = ax.scatter(
    acceleration_position_window_df["x"],
    acceleration_position_window_df["y"],
    c=acceleration_position_window_df["acceleration"],
    s=14,
    cmap="coolwarm",
    vmin=-accel_abs_max,
    vmax=accel_abs_max,
    zorder=3,
)
ax.scatter(
    acceleration_position_window_df["x"].iloc[0],
    acceleration_position_window_df["y"].iloc[0],
    color=COLORS["green"],
    edgecolor="white",
    linewidth=0.8,
    s=70,
    zorder=4,
    label="start",
)
ax.scatter(
    acceleration_position_window_df["x"].iloc[-1],
    acceleration_position_window_df["y"].iloc[-1],
    color=COLORS["orange"],
    edgecolor="white",
    linewidth=0.8,
    s=70,
    zorder=4,
    label="stop",
)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x position")
ax.set_ylabel("y position")
plt.colorbar(sc, ax=ax, label="acceleration (cm/s^2)")
ax.legend(frameon=False, loc="best")
sns.despine(ax=ax)
plt.show()

## 6. Inspect Units And Brain Regions

Neuropixels probes record along a long, thin shank that can pass through several structures on its way to the target region. Even in a session aimed at MEC and CA1, some recorded units may end up assigned to a neighboring structure, such as dentate gyrus, CA3, or visual cortex, depending on exactly where each electrode sits along the shank. The `units` table in this dataset does not label each neuron by region, but each unit is linked to the electrode(s) that recorded it, and the `electrodes` table records the anatomical location of each electrode. Combining the two tells us which region each unit came from.

First, inspect the size of the recording table. The `units` table tells us how many neurons were sorted, and the `electrodes` table tells us where the recording channels were located.

In [ ]:
units_table = nwb.units
unit_ids = np.asarray(units_table.id[:])
electrodes_df = nwb.electrodes.to_dataframe()

print(f"Number of units: {len(unit_ids)}")
print(f"Number of electrodes: {len(electrodes_df)}")
print("Unit columns:", list(units_table.colnames))

if "location" in electrodes_df.columns:
    display(electrodes_df["location"].value_counts().rename("n_electrodes").to_frame())
else:
    print("No location column found in electrodes table.")


Next, count how many units are assigned to each brain region. 

In [ ]:
# Use small helpers from tutorial_utils.py so this cell stays focused on the table we want.
def get_unit_spike_times(unit_id):
    return np.asarray(fetch_unit_spike_times(units_table, unit_ids, unit_id))


# spike_times_index stores cumulative spike counts, so this reads only a small index vector,
# not every spike time in the session.
spike_time_ends = np.asarray(units_table["spike_times_index"].data[:])
total_spikes = np.diff(np.r_[0, spike_time_ends])
regions = [unit_region_from_row(units_table, row_i, electrodes_df) for row_i in range(len(unit_ids))]

units_df = pd.DataFrame(
    {
        "unit_id": unit_ids,
        "region": regions,
        "total_spikes": total_spikes,
    },
    index=unit_ids,
)
units_df.index.name = "unit_id"

display(units_df["region"].value_counts().rename("n_units").to_frame())

We will focus on CA1 for the spatial firing analysis.

CA1 is the classic region for studying place cells, first described by O'Keefe and Dostrovsky in 1971. 

In [31]:
region_filter = "CA1"

region_units = units_df.query("region == @region_filter").copy()
region_units["firing_rate_hz"] = region_units["total_spikes"] / (time[-1] - time[0])

median_rate_hz = region_units["firing_rate_hz"].median()
region_units["rate_distance"] = (region_units["firing_rate_hz"] - median_rate_hz).abs()

print(f"Region: {region_filter}")
print(f"Units in region: {len(region_units)}")
print(f"Median firing rate: {median_rate_hz:.2f} Hz")
display(
    region_units[["region", "total_spikes", "firing_rate_hz"]]
    .loc[region_units.sort_values("rate_distance").head(15).index]
    .sort_values("firing_rate_hz")
)

Region: CA1
Units in region: 338
Median firing rate: 1.00 Hz


,region,total_spikes,firing_rate_hz
unit_id,,,
9,CA1,2195,0.897542
523,CA1,2196,0.897951
347,CA1,2273,0.929436
290,CA1,2304,0.942112
105,CA1,2351,0.961331
293,CA1,2394,0.978913
10,CA1,2435,0.995678
245,CA1,2460,1.005901
589,CA1,2478,1.013261


## 7. Select Candidate Units

Before plotting spike locations, predict whether a well-tuned cell will fire everywhere or in specific parts of the maze.

Firing rate alone can separate broad cell classes (interneurons vs. pyramidal cells), but says nothing about whether a pyramidal cell's spikes are actually organized by location. We first exclude likely interneurons using a 10 Hz cutoff (this session's CA1 firing-rate histogram has a clear dip around 9-12 Hz separating a low-rate group from a smaller, much faster-firing group). We then rank the remaining candidates directly by spatial information content, in bits per spike (Skaggs et al., 1993):

```text
info (bits/spike) = sum over bins i of: p_i * (r_i / r_mean) * log2(r_i / r_mean)
```

where `p_i` is the fraction of time spent in bin `i`, `r_i` is the firing rate in bin `i`, and `r_mean` is the overall mean firing rate. A cell that fires at the same rate everywhere gets an information score of zero no matter how active it is; a high score means firing is concentrated in a small fraction of the space relative to how much time the animal spent there. This is a direct measure of spatial tuning, unlike firing rate or spike count.

The table below lists the 20 CA1 units (excluding interneurons) with the highest spatial information. We will look at their spike timing and rate maps next, before picking one to carry through the rest of the notebook.

In [ ]:
interneuron_rate_hz = 10.0  # dip in the CA1 firing-rate histogram sits around 9-12 Hz
min_rate_hz = 0.1  # below this, too few spikes to trust the info estimate

n_bins = 40
min_occupancy = 0.1
occupancy, x_edges, y_edges = np.histogram2d(
    x[:-1], y[:-1], bins=n_bins, weights=dt
)
occupied = occupancy > min_occupancy
occupancy_p = occupancy / occupancy.sum()

rate_filtered_units = region_units[
    (region_units["firing_rate_hz"] >= min_rate_hz) & (region_units["firing_rate_hz"] < interneuron_rate_hz)
]

spatial_info_results = []
for candidate_id, candidate_row in rate_filtered_units.iterrows():
    st = get_unit_spike_times(candidate_id)
    st = st[(st >= time[0]) & (st <= time[-1])]
    if len(st) < 100:
        continue
    cx = np.interp(st, time, x)
    cy = np.interp(st, time, y)

    counts, _, _ = np.histogram2d(cx, cy, bins=[x_edges, y_edges])
    candidate_rate_map = np.zeros_like(occupancy)
    candidate_rate_map[occupied] = counts[occupied] / occupancy[occupied]

    mean_rate = np.sum(candidate_rate_map[occupied] * occupancy_p[occupied]) / np.sum(occupancy_p[occupied])
    if mean_rate <= 0:
        continue

    r = candidate_rate_map[occupied]
    p = occupancy_p[occupied]
    firing = r > 0
    info_bits_per_spike = np.sum(p[firing] * (r[firing] / mean_rate) * np.log2(r[firing] / mean_rate))

    spatial_info_results.append(
        {
            "unit_id": candidate_id,
            "total_spikes": candidate_row["total_spikes"],
            "firing_rate_hz": candidate_row["firing_rate_hz"],
            "peak_to_mean": candidate_rate_map.max() / mean_rate,
            "info_bits_per_spike": info_bits_per_spike,
        }
    )

n_candidates = 20
candidate_units = pd.DataFrame(spatial_info_results).set_index("unit_id")
candidate_units = candidate_units.sort_values("info_bits_per_spike", ascending=False).head(n_candidates)
display(candidate_units.round(3))


**Scan the candidates:** drag the slider to flip through each unit's rate map, ordered from highest to lowest spatial information.

In [ ]:
import ipywidgets as widgets
from scipy.ndimage import gaussian_filter

smoothing_sigma = 1  # in bins


def plot_candidate_rate_map(unit_id):
    row = candidate_units.loc[unit_id]

    st = get_unit_spike_times(unit_id)
    st = st[(st >= time[0]) & (st <= time[-1])]
    cx = np.interp(st, time, x)
    cy = np.interp(st, time, y)

    counts, _, _ = np.histogram2d(cx, cy, bins=[x_edges, y_edges])
    smoothed_counts = gaussian_filter(counts, sigma=smoothing_sigma)
    smoothed_occupancy = gaussian_filter(occupancy, sigma=smoothing_sigma)
    smoothed_occupied = smoothed_occupancy > min_occupancy

    smoothed_rate_map = np.full_like(occupancy, np.nan)
    smoothed_rate_map[smoothed_occupied] = (
        smoothed_counts[smoothed_occupied] / smoothed_occupancy[smoothed_occupied]
    )

    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(
        smoothed_rate_map.T,
        origin="lower",
        extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
        aspect="equal",
        cmap="viridis",
    )
    ax.set_xlabel("x position")
    ax.set_ylabel("y position")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="spikes/s")
    sns.despine(ax=ax)
    plt.show()

    print(
        f"unit_id: {unit_id}  |  firing rate: {row['firing_rate_hz']:.2f} Hz  |  "
        f"total spikes: {int(row['total_spikes'])}  |  "
        f"peak/mean: {row['peak_to_mean']:.1f}  |  "
        f"info: {row['info_bits_per_spike']:.2f} bits/spike"
    )


unit_slider = widgets.SelectionSlider(
    options=list(candidate_units.index),
    description="unit_id",
    continuous_update=False,
)
widgets.interact(plot_candidate_rate_map, unit_id=unit_slider)


## 8. Raster Plot For The Candidate Units

A raster plot shows the spike times of many neurons at once, with each row a different unit and each tick mark a single spike. Looking at a short window (tens of seconds) makes it possible to see the fine timing of spikes relative to each other and to behavior, something that would be an unreadable blur over the full session, which runs about 40 minutes. The raster below shows only the 20 candidate units from the table above, so you can see their raw spike timing before picking one. A short window like this is useful for visualizing spike timing, but is not enough on its own to say whether a cell is spatially tuned, since the animal may only have visited a small part of the maze during that window.

In [ ]:
raster_start_time = 300.0
raster_stop_time = 360.0

fig, ax = plt.subplots(figsize=(9, 5))
for row_i, unit_id in enumerate(candidate_units.index):
    spikes = get_unit_spike_times(unit_id)
    spikes = spikes[(spikes >= raster_start_time) & (spikes <= raster_stop_time)]
    ax.vlines(spikes, row_i + 0.1, row_i + 0.9, color=COLORS["dark"], linewidth=0.6)

ax.set_xlabel("time (s)")
ax.set_ylabel("unit_id")
ax.set_yticks(np.arange(len(candidate_units)) + 0.5)
ax.set_yticklabels(candidate_units.index)
ax.set_title(f"Raster: top {len(candidate_units)} candidate units by spatial information")
ax.set_xlim(raster_start_time, raster_stop_time)
sns.despine(ax=ax)
plt.show()


**Questions**

- Are spikes evenly distributed across time?
- Do some cells fire much more than others?
- Why is a short window useful for a raster but not enough for estimating a place field?

**Decision:** pick one `unit_id` from the table above.

In [ ]:
# Choose a unit_id from the candidate table above.
unit_id = 125

unit_spike_times = get_unit_spike_times(unit_id)
unit_spike_times = unit_spike_times[(unit_spike_times >= time[0]) & (unit_spike_times <= time[-1])]

print("unit_id:", unit_id)
print("region:", units_df.loc[unit_id, "region"])
print("firing rate (Hz):", round(region_units.loc[unit_id, "firing_rate_hz"], 2))
print("total spikes in session:", len(unit_spike_times))


## 9. Spike Locations On The Maze

Spike times and position samples are measured on different time grids. Interpolation estimates where the animal was at each spike time.

In [ ]:
spike_x = np.interp(unit_spike_times, time, x)
spike_y = np.interp(unit_spike_times, time, y)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(x, y, color=COLORS["light_gray"], linewidth=0.7, zorder=1)
ax.scatter(spike_x, spike_y, s=2, color=COLORS["vermillion"], alpha=0.75, zorder=2)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.set_title(f"Spike locations for unit {unit_id}")
sns.despine(ax=ax)
plt.show()


**Questions**

- Are spike locations clustered?
- Could clustering be explained by where the animal spent more time?

### Occupancy

The spike-location plot above shows where this unit fired, but not where the animal spent time. A cluster of spikes in one arm of the maze could mean the neuron prefers that location, or it could simply mean the animal lingered there. Spike locations alone cannot distinguish these two explanations, so they are not enough evidence for a place cell on their own.

Occupancy is the amount of time the animal spent in each spatial bin. Correcting for it is what turns a spike-location plot into a firing-rate estimate:

```text
firing rate at position = spikes at position / time spent at position
```

In [ ]:
n_bins = 40

occupancy, x_edges, y_edges = np.histogram2d(
    x[:-1],
    y[:-1],
    bins=n_bins,
    weights=dt,
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    occupancy.T,
    origin="lower",
    extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
    aspect="equal",
    cmap="mako",
)
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.set_title("Occupancy map")
plt.colorbar(im, ax=ax, label="seconds")
sns.despine(ax=ax)
plt.show()


**Questions**

- Compare this occupancy map to the spike-location plot above. Do the densest spike clusters line up with the most-visited locations?
- Given that comparison, would you trust the spike-location plot alone as evidence this is a place cell? Why or why not?

## 10. Rate Maps

A rate map estimates how strongly a neuron fires at each location. It is more interpretable than raw spike locations because it corrects for unequal sampling of the maze.

In [ ]:
spike_counts, _, _ = np.histogram2d(spike_x, spike_y, bins=[x_edges, y_edges])

min_occupancy = 0.1
rate_map = np.full_like(occupancy, np.nan, dtype=float)
valid = occupancy > min_occupancy
rate_map[valid] = spike_counts[valid] / occupancy[valid]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)

maps = [occupancy, spike_counts, rate_map]
titles = ["Occupancy (s)", "Spike count", "Firing rate (spikes/s)"]
cmaps = ["mako", "rocket", "viridis"]

for ax, data, title, cmap in zip(axes, maps, titles, cmaps):
    im = ax.imshow(
        data.T,
        origin="lower",
        extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
        aspect="equal",
        cmap=cmap,
    )
    ax.set_title(title)
    ax.set_xlabel("x position")
    ax.set_ylabel("y position")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    sns.despine(ax=ax)

plt.show()


In [ ]:
from scipy.ndimage import gaussian_filter

smoothing_sigma = 1 # in bins

# Smooth the spike counts and occupancy separately, then divide, so that
# unvisited (NaN) bins don't leak into their smoothed neighbors.
smoothed_spike_counts = gaussian_filter(spike_counts, sigma=smoothing_sigma)
smoothed_occupancy = gaussian_filter(occupancy, sigma=smoothing_sigma)

smoothed_rate_map = np.full_like(rate_map, np.nan)
valid_smoothed = smoothed_occupancy > min_occupancy
smoothed_rate_map[valid_smoothed] = (
    smoothed_spike_counts[valid_smoothed] / smoothed_occupancy[valid_smoothed]
)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(
    smoothed_rate_map.T,
    origin="lower",
    extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
    aspect="equal",
    cmap="viridis",
)
ax.set_xlabel("x position")
ax.set_ylabel("y position")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="spikes/s")
sns.despine(ax=ax)
plt.show()

**Interpretation questions**

- Did the rate map change your interpretation of the spike-location plot?
- Does this unit look spatially modulated?
- What would make the conclusion stronger?

## 12. LFP Preview

If time remains, inspect the selected-channel LFP. This file contains LFP from selected channels, not every Neuropixels channel.

In [ ]:
lfp_paths = []
for module_name, module in nwb.processing.items():
    if "LFP" in module.data_interfaces:
        lfp_paths.append(module_name)

print("Processing modules with LFP:")
print(lfp_paths)

In [ ]:
if lfp_paths:
    lfp_module = lfp_paths[0]
    lfp_series = nwb.processing[lfp_module]["LFP"]["LFP"]

    lfp_start_time = 300.0
    lfp_stop_time = 310.0
    lfp_rate = lfp_series.rate
    lfp_start = int((lfp_start_time - lfp_series.starting_time) * lfp_rate)
    lfp_stop = int((lfp_stop_time - lfp_series.starting_time) * lfp_rate)

    lfp_data = np.asarray(lfp_series.data[lfp_start:lfp_stop, 0])
    lfp_time = lfp_series.starting_time + np.arange(lfp_start, lfp_stop) / lfp_rate

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(lfp_time, lfp_data, color=COLORS["purple"], linewidth=0.8)
    ax.set_xlabel("time (s)")
    ax.set_ylabel(lfp_series.unit)
    ax.set_title(f"LFP preview: {lfp_module}")
    sns.despine(ax=ax)
    plt.show()
else:
    print("No LFP module found in this file.")

## Wrap-Up

By this point, you have streamed an open NWB file, inspected its structure, extracted position, computed movement variables and occupancy, selected a brain region and unit, and asked whether firing relates to task-structured space.

For a group report, fill in:

| Group | Subject/session | Region | Unit ID | Spatial? | Evidence |
| --- | --- | --- | ---: | --- | --- |
|  |  |  |  |  |  |